# A05. 베르누이 시행과 이항분포

**학습 목표**
- 베르누이 시행(Bernoulli Trial)의 개념과 `scipy.stats.bernoulli` 사용법을 익힌다
- 이항분포(Binomial Distribution) $B(n, p)$의 PMF를 이해하고 시각화한다
- 시뮬레이션을 통해 이론적 분포와 실험 결과를 비교한다
- 편향된 동전($p=0.7$)에서 베르누이, 이항, 정규분포를 비교한다

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import binom, bernoulli, norm, uniform, expon, pareto
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
sns.set_palette('Set2')

np.random.seed(42)
print('라이브러리 로드 완료')

---
## 1. 베르누이 시행 (Bernoulli Trial)

### 핵심 용어 정리

| 용어 | 정의 | 예시 |
|------|------|------|
| **시행(Trial)** | 결과를 관찰하는 실험 1회 | 동전 1번 던지기 |
| **성공(Success)** | 관심 있는 결과 | 앞면(Head) |
| **실패(Failure)** | 나머지 결과 | 뒷면(Tail) |
| **성공확률 $p$** | 한 번의 시행에서 성공할 확률 | $p = 0.5$ (공정한 동전) |
| **실패확률 $q$** | $q = 1 - p$ | $q = 0.5$ |

### 베르누이 확률변수

$$X \sim \text{Bernoulli}(p)$$

$$P(X = x) = p^x (1-p)^{1-x}, \quad x \in \{0, 1\}$$

- $X = 1$: 성공 (확률 $p$)
- $X = 0$: 실패 (확률 $1 - p$)
- **기댓값**: $E[X] = p$
- **분산**: $\text{Var}(X) = p(1-p)$

In [ ]:
# === 베르누이 분포: scipy.stats.bernoulli 사용법 ===
p = 0.5  # 공정한 동전

# 베르누이 분포 객체 생성
rv_bernoulli = bernoulli(p)

# 기본 통계량
print('=== 공정한 동전 베르누이 분포 (p=0.5) ===')
print(f'P(X=0) = {rv_bernoulli.pmf(0):.4f}  (뒷면)')
print(f'P(X=1) = {rv_bernoulli.pmf(1):.4f}  (앞면)')
print(f'기댓값 E[X] = {rv_bernoulli.mean():.4f}')
print(f'분  산 Var(X) = {rv_bernoulli.var():.4f}')
print(f'표준편차 σ = {rv_bernoulli.std():.4f}')

# 베르누이 랜덤 샘플 10개
samples = rv_bernoulli.rvs(size=10)
print(f'\n랜덤 샘플 10개: {samples}')
print(f'→ 앞면 {sum(samples)}번, 뒷면 {10 - sum(samples)}번')

In [ ]:
# === 베르누이 시행 시각화 ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (1) 공정한 동전 (p=0.5)
x = [0, 1]
probs_fair = [bernoulli.pmf(0, 0.5), bernoulli.pmf(1, 0.5)]
colors = sns.color_palette('Set2', 2)
axes[0].bar(x, probs_fair, color=colors, width=0.4, edgecolor='black', linewidth=1.2)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['X=0\n(뒷면)', 'X=1\n(앞면)'], fontsize=12)
axes[0].set_ylabel('확률 P(X)', fontsize=12)
axes[0].set_title('공정한 동전: Bernoulli(p=0.5)', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 0.8)
for i, v in enumerate(probs_fair):
    axes[0].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=14, fontweight='bold')

# (2) 편향된 동전 (p=0.7)
probs_biased = [bernoulli.pmf(0, 0.7), bernoulli.pmf(1, 0.7)]
axes[1].bar(x, probs_biased, color=colors, width=0.4, edgecolor='black', linewidth=1.2)
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['X=0\n(뒷면)', 'X=1\n(앞면)'], fontsize=12)
axes[1].set_ylabel('확률 P(X)', fontsize=12)
axes[1].set_title('찌그러진 동전: Bernoulli(p=0.7)', fontsize=14, fontweight='bold')
axes[1].set_ylim(0, 0.8)
for i, v in enumerate(probs_biased):
    axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### 해석

- **공정한 동전** ($p=0.5$): 앞면과 뒷면의 확률이 동일하게 0.5로 대칭적이다.
- **찌그러진 동전** ($p=0.7$): 앞면이 나올 확률이 0.7로, 뒷면(0.3)보다 2배 이상 높다. 이처럼 $p \neq 0.5$인 경우를 **편향된(biased)** 베르누이 시행이라 한다.

---
## 2. 이항분포 (Binomial Distribution)

### 정의

베르누이 시행을 **독립적으로 $n$번 반복**할 때, 성공 횟수 $k$의 분포:

$$X \sim B(n, p)$$

### 확률질량함수 (PMF)

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}, \quad k = 0, 1, 2, \ldots, n$$

### 이항계수 (Binomial Coefficient)

$$\binom{n}{k} = C(n, k) = \frac{n!}{k!(n-k)!}$$

이항계수 $\binom{n}{k}$는 $n$개 중 $k$개를 선택하는 **조합의 수**이다.

예: $\binom{10}{3} = \frac{10!}{3! \cdot 7!} = 120$

### 기댓값과 분산

- **기댓값**: $E[X] = np$
- **분산**: $\text{Var}(X) = np(1-p)$
- **표준편차**: $\sigma = \sqrt{np(1-p)}$

In [ ]:
# === 이항분포 B(n=10, p=0.5) PMF 시각화 ===
n, p = 10, 0.5
k = np.arange(0, n + 1)
pmf_values = binom.pmf(k, n, p)

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.bar(k, pmf_values, color=sns.color_palette('Set2')[0], 
              edgecolor='black', linewidth=1.2, alpha=0.85, label='PMF')

# 각 막대 위에 확률값 표시
for i, (ki, pi) in enumerate(zip(k, pmf_values)):
    ax.text(ki, pi + 0.005, f'{pi:.4f}', ha='center', fontsize=9, fontweight='bold')

# 기댓값 표시
mean = n * p
ax.axvline(mean, color='red', linestyle='--', linewidth=2, label=f'기댓값 E[X] = {mean:.1f}')

ax.set_xlabel('성공 횟수 k (앞면 개수)', fontsize=13)
ax.set_ylabel('확률 P(X=k)', fontsize=13)
ax.set_title(f'이항분포 B(n={n}, p={p}) 의 확률질량함수 (PMF)', fontsize=15, fontweight='bold')
ax.set_xticks(k)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

# 이항계수 예시
from math import comb
print('\n=== 이항계수 C(10, k) ===')
for ki in k:
    print(f'  C(10, {ki:2d}) = {comb(10, ki):>4d}  →  P(X={ki:2d}) = {binom.pmf(ki, n, p):.6f}')

### 해석

- $B(10, 0.5)$는 **완벽한 좌우 대칭** 형태를 보인다. 이는 $p = 0.5$이기 때문이다.
- **가장 높은 확률**: $P(X=5) = 0.2461$ → 10번 던져서 앞면 5번이 가장 가능성이 높다.
- 이항계수 $C(10, k)$가 $k=5$에서 최대(252)이므로, 조합의 수가 확률에 직접 영향을 준다.
- 기댓값 $E[X] = np = 10 \times 0.5 = 5$로, 분포의 중심에 위치한다.

---
## 3. 시뮬레이션: 1000번 실험 vs 이론 PMF

In [ ]:
# === 1000번 시뮬레이션: 동전 10번 던지기를 1000번 반복 ===
n, p = 10, 0.5
n_simulations = 1000

# 시뮬레이션 수행
sim_results = binom.rvs(n, p, size=n_simulations)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (1) 히스토그램: 시뮬레이션 결과
k = np.arange(0, n + 1)
axes[0].hist(sim_results, bins=np.arange(-0.5, n + 1.5, 1), 
             density=True, color=sns.color_palette('Set2')[1], 
             edgecolor='black', alpha=0.7, label='시뮬레이션 (1000회)')
axes[0].plot(k, binom.pmf(k, n, p), 'ro-', markersize=8, linewidth=2, 
             label='이론 PMF', zorder=5)
axes[0].set_xlabel('앞면 개수 k', fontsize=13)
axes[0].set_ylabel('상대 빈도 / 확률', fontsize=13)
axes[0].set_title('시뮬레이션 1,000회 vs 이론 PMF', fontsize=14, fontweight='bold')
axes[0].set_xticks(k)
axes[0].legend(fontsize=11)

# (2) 누적 빈도: 시뮬레이션 횟수 증가에 따른 수렴
cumulative_means = np.cumsum(sim_results) / np.arange(1, n_simulations + 1)
axes[1].plot(range(1, n_simulations + 1), cumulative_means, 
             color=sns.color_palette('Set2')[2], linewidth=1.5, alpha=0.8)
axes[1].axhline(y=n * p, color='red', linestyle='--', linewidth=2, 
                label=f'이론적 기댓값 = {n*p:.1f}')
axes[1].set_xlabel('시뮬레이션 횟수', fontsize=13)
axes[1].set_ylabel('누적 평균', fontsize=13)
axes[1].set_title('큰 수의 법칙: 누적 평균의 수렴', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].set_ylim(3, 7)

plt.tight_layout()
plt.show()

# 통계 비교
print('=== 시뮬레이션 vs 이론 비교 ===')
print(f'시뮬레이션 평균: {np.mean(sim_results):.4f}  vs  이론 기댓값: {n*p:.4f}')
print(f'시뮬레이션 분산: {np.var(sim_results):.4f}  vs  이론 분산: {n*p*(1-p):.4f}')

### 해석

- **왼쪽 그래프**: 1000번 시뮬레이션의 히스토그램(막대)과 이론적 PMF(빨간 점)가 매우 잘 일치한다. 시뮬레이션 횟수가 충분하면 실험 결과가 이론에 수렴함을 보여준다.
- **오른쪽 그래프**: **큰 수의 법칙(Law of Large Numbers)**에 의해 시뮬레이션 횟수가 늘어날수록 누적 평균이 이론적 기댓값(5.0)에 수렴한다.
- 초반에는 변동이 크지만, 약 200~300회 이후부터는 거의 5.0에 안정적으로 수렴한다.

---
## 4. 이항분포 100번 실험하면 어떤 그래프?

In [ ]:
# === n=100일 때의 이항분포 → 정규분포에 근사 ===
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, n_val in enumerate([10, 50, 100]):
    p_val = 0.5
    k_range = np.arange(0, n_val + 1)
    pmf_vals = binom.pmf(k_range, n_val, p_val)
    
    # 이항분포 PMF
    axes[idx].bar(k_range, pmf_vals, color=sns.color_palette('Set2')[0], 
                  alpha=0.7, edgecolor='gray', linewidth=0.5, label=f'B({n_val}, {p_val})')
    
    # 정규분포 근사 곡선
    mu = n_val * p_val
    sigma = np.sqrt(n_val * p_val * (1 - p_val))
    x_norm = np.linspace(k_range[0], k_range[-1], 200)
    axes[idx].plot(x_norm, norm.pdf(x_norm, mu, sigma), 'r-', 
                   linewidth=2.5, label=f'N({mu:.0f}, {sigma:.2f}²)')
    
    axes[idx].set_xlabel('성공 횟수 k', fontsize=12)
    axes[idx].set_ylabel('확률', fontsize=12)
    axes[idx].set_title(f'B(n={n_val}, p={p_val})', fontsize=14, fontweight='bold')
    axes[idx].legend(fontsize=10)

plt.suptitle('이항분포 → 정규분포 근사 (n이 커질수록)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 해석

- $n=10$: 이항분포의 막대가 뚜렷하게 보이며, 정규분포 곡선과 약간의 차이가 있다.
- $n=50$: 막대가 촘촘해지면서 정규분포 곡선에 매우 잘 근사한다.
- $n=100$: 사실상 **정규분포와 구별할 수 없는 수준**으로 근사한다.

> **중심극한정리(CLT)**: $n$이 충분히 크면 이항분포 $B(n, p)$는 정규분포 $N(np, np(1-p))$에 근사한다.

---
## 5. 편향된 동전 (p=0.7): 3가지 분포 비교

찌그러진 동전(앞면이 나올 확률 $p=0.7$)에 대해 **베르누이, 이항, 정규분포**를 비교한다.

In [ ]:
# === 편향된 동전 p=0.7: 3가지 분포 비교 ===
p_biased = 0.7
n_trials = 10

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- (1) 베르누이 분포: Bernoulli(p=0.7) ---
x_bern = [0, 1]
pmf_bern = [bernoulli.pmf(0, p_biased), bernoulli.pmf(1, p_biased)]
colors = [sns.color_palette('Set2')[1], sns.color_palette('Set2')[0]]
axes[0].bar(x_bern, pmf_bern, color=colors, width=0.4, edgecolor='black', linewidth=1.2)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['X=0 (뒷면)', 'X=1 (앞면)'], fontsize=11)
axes[0].set_ylabel('확률', fontsize=12)
axes[0].set_title(f'① 베르누이 분포\nBernoulli(p={p_biased})', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 0.9)
for i, v in enumerate(pmf_bern):
    axes[0].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=14, fontweight='bold')

# --- (2) 이항분포: B(n=10, p=0.7) ---
k = np.arange(0, n_trials + 1)
pmf_binom = binom.pmf(k, n_trials, p_biased)
bar_colors = [sns.color_palette('Set2')[0] if ki >= 7 else sns.color_palette('Set2')[1] for ki in k]
axes[1].bar(k, pmf_binom, color=bar_colors, edgecolor='black', linewidth=0.8)
axes[1].set_xlabel('성공 횟수 k', fontsize=12)
axes[1].set_ylabel('확률 P(X=k)', fontsize=12)
axes[1].set_title(f'② 이항분포\nB(n={n_trials}, p={p_biased})', fontsize=13, fontweight='bold')
axes[1].set_xticks(k)
mu_binom = n_trials * p_biased
axes[1].axvline(mu_binom, color='red', linestyle='--', linewidth=2, label=f'E[X]={mu_binom:.1f}')
axes[1].legend(fontsize=11)

# --- (3) 정규분포 근사: N(np, np(1-p)) ---
mu_norm = n_trials * p_biased
sigma_norm = np.sqrt(n_trials * p_biased * (1 - p_biased))
x_norm = np.linspace(0, n_trials, 300)
pdf_norm = norm.pdf(x_norm, mu_norm, sigma_norm)

axes[2].bar(k, pmf_binom, color=sns.color_palette('Set2')[1], alpha=0.5, 
            edgecolor='gray', linewidth=0.8, label=f'B({n_trials}, {p_biased})')
axes[2].plot(x_norm, pdf_norm, 'r-', linewidth=2.5, 
             label=f'N({mu_norm:.1f}, {sigma_norm:.2f}²)')
axes[2].fill_between(x_norm, pdf_norm, alpha=0.15, color='red')
axes[2].set_xlabel('성공 횟수 k', fontsize=12)
axes[2].set_ylabel('확률/밀도', fontsize=12)
axes[2].set_title(f'③ 이항 vs 정규 근사\n$\mu$={mu_norm:.1f}, $\sigma$={sigma_norm:.2f}', 
                   fontsize=13, fontweight='bold')
axes[2].set_xticks(k)
axes[2].legend(fontsize=10)

plt.suptitle(f'편향된 동전 (p={p_biased}): 베르누이 → 이항 → 정규 비교', 
             fontsize=16, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

# 통계량 비교 테이블
print('\n=== 편향된 동전 (p=0.7) 통계량 비교 ===')
comparison = pd.DataFrame({
    '분포': ['Bernoulli(0.7)', 'B(10, 0.7)', f'N({mu_norm}, {sigma_norm:.2f}²)'],
    '기댓값': [p_biased, n_trials * p_biased, mu_norm],
    '분산': [p_biased*(1-p_biased), n_trials*p_biased*(1-p_biased), sigma_norm**2],
    '표준편차': [np.sqrt(p_biased*(1-p_biased)), sigma_norm, sigma_norm]
})
print(comparison.to_string(index=False))

### 해석

- **① 베르누이**: 1회 시행의 기본 구조. $p=0.7$이므로 앞면(성공) 확률이 뒷면보다 높다.
- **② 이항분포**: 베르누이 시행을 10번 반복. 분포가 **왼쪽으로 치우친(left-skewed)** 형태를 보인다. $p > 0.5$이므로 성공 횟수가 높은 쪽에 무게가 실린다. 기댓값 $E[X] = 7$.
- **③ 정규 근사**: $n=10$에서도 정규분포 곡선이 이항분포를 상당히 잘 근사한다. 다만 비대칭성이 있어 $n$이 더 커야 완벽한 근사가 된다.

> **핵심**: 베르누이 → 이항 → 정규분포는 **동일한 현상을 다른 스케일로 바라보는 것**이다.

---
## 6. 심화: 다양한 p 값에 따른 이항분포 비교

In [ ]:
# === 다양한 p 값에 따른 이항분포 형태 변화 ===
n_val = 20
p_values = [0.1, 0.3, 0.5, 0.7, 0.9]
k_range = np.arange(0, n_val + 1)

fig, ax = plt.subplots(figsize=(14, 7))
colors = sns.color_palette('husl', len(p_values))

for i, p_val in enumerate(p_values):
    pmf_vals = binom.pmf(k_range, n_val, p_val)
    ax.plot(k_range, pmf_vals, 'o-', color=colors[i], markersize=6, 
            linewidth=2, label=f'B({n_val}, {p_val}) | E[X]={n_val*p_val:.0f}', alpha=0.85)

ax.set_xlabel('성공 횟수 k', fontsize=13)
ax.set_ylabel('확률 P(X=k)', fontsize=13)
ax.set_title(f'이항분포 B(n={n_val}, p)에서 p 값에 따른 형태 변화', fontsize=15, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.set_xticks(k_range)
plt.tight_layout()
plt.show()

### 해석

- **$p = 0.5$**: 완벽한 좌우 대칭. 기댓값이 $n/2 = 10$에 위치.
- **$p < 0.5$ (0.1, 0.3)**: 오른쪽 꼬리가 긴 **양의 왜도(right-skewed)**. $p$가 작을수록 왜도가 커진다.
- **$p > 0.5$ (0.7, 0.9)**: 왼쪽 꼬리가 긴 **음의 왜도(left-skewed)**.
- $p = 0.1$과 $p = 0.9$는 **거울 대칭** 관계이다: $B(n, p)$와 $B(n, 1-p)$는 좌우가 뒤집힌 형태.

---
## 7. 정리

| 분포 | 수식 | 기댓값 | 분산 | 특징 |
|------|------|--------|------|------|
| 베르누이 | $P(X=x) = p^x(1-p)^{1-x}$ | $p$ | $p(1-p)$ | 1회 시행 |
| 이항분포 | $P(X=k) = \binom{n}{k}p^k(1-p)^{n-k}$ | $np$ | $np(1-p)$ | $n$번 독립 반복 |
| 정규근사 | $N(np, np(1-p))$ | $np$ | $np(1-p)$ | $n$ 충분히 클 때 |

**핵심 요약**:
1. **베르누이 시행**은 성공/실패 두 가지 결과만 갖는 **가장 기본적인** 확률 실험이다.
2. **이항분포**는 베르누이 시행을 $n$번 독립 반복했을 때의 성공 횟수 분포이다.
3. $n$이 충분히 크면 이항분포는 **정규분포에 근사**한다 (중심극한정리).
4. $p = 0.5$일 때 대칭, $p \neq 0.5$일 때 비대칭(왜도 발생).
5. `scipy.stats`의 `bernoulli`와 `binom` 모듈로 손쉽게 계산하고 시뮬레이션할 수 있다.